In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.model_selection import GroupShuffleSplit

sys.path.append(os.path.abspath('..'))
from src.data_preprocessing import get_modeling_datasets
from src.models import PALModel

# 1. Load the STRICTLY SEPARATED modeling datasets
train_imp, test_imp, dqa_imp = get_modeling_datasets()

print(f"Standard Train Size: {len(train_imp)}")
print(f"Standard Test Size: {len(test_imp)}")
print(f"DQA Subset Size: {len(dqa_imp)}")

In [ ]:
# Create GroupShuffleSplit object
gss_train_valtest = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
gss_val_test = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42) # Splits the 20% into 10/10

# 1. Split Train vs (Val + Test) based on session_idx
train_idx, valtest_idx = next(gss_train_valtest.split(dqa_imp, groups=dqa_imp['session_idx']))
dqa_train = dqa_imp.iloc[train_idx].copy()
dqa_valtest = dqa_imp.iloc[valtest_idx].copy()

# 2. Split Val vs Test
val_idx, test_idx = next(gss_val_test.split(dqa_valtest, groups=dqa_valtest['session_idx']))
dqa_val = dqa_valtest.iloc[val_idx].copy()
dqa_test = dqa_valtest.iloc[test_idx].copy()

# Add a dummy DQA feature column (Always 1 for this subset)
for df in [dqa_train, dqa_val, dqa_test]:
    df['dqa_present'] = 1

print(f"DQA Train: {len(dqa_train)} | Val: {len(dqa_val)} | Test: {len(dqa_test)}")

In [ ]:
def evaluate_lr(X_train, y_train, X_test, y_test, name="Baseline"):
    lr = LogisticRegression()
    lr.fit(X_train, y_train)
    probs = lr.predict_proba(X_test)[:, 1]
    
    auc = roc_auc_score(y_test, probs)
    loss = log_loss(y_test, probs)
    print(f"--- {name} Logistic Regression ---")
    print(f"ROC-AUC: {auc:.4f} | LogLoss: {loss:.4f}\n")
    return lr

# Standard LR Baseline
X_train_std = train_imp[['position', 'is_left_column']].fillna(0)
y_train_std = train_imp['click'].fillna(0)
X_test_std = test_imp[['position', 'is_left_column']].fillna(0)
y_test_std = test_imp['click'].fillna(0)

lr_std = evaluate_lr(X_train_std, y_train_std, X_test_std, y_test_std, "Standard")

# DQA LR Baseline
X_train_dqa = dqa_train[['position', 'is_left_column']].fillna(0)
y_train_dqa = dqa_train['click'].fillna(0)
X_test_dqa = dqa_test[['position', 'is_left_column']].fillna(0)
y_test_dqa = dqa_test['click'].fillna(0)

lr_dqa = evaluate_lr(X_train_dqa, y_train_dqa, X_test_dqa, y_test_dqa, "DQA-Present")

In [ ]:
def train_pal_model(model, train_df, epochs=5, lr=0.01, use_dqa=False):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss() # Binary Cross Entropy for CTR prediction
    
    # Convert data to tensors
    pos_t = torch.tensor(train_df['position'].values, dtype=torch.long)
    left_t = torch.tensor(train_df['is_left_column'].astype(int).values, dtype=torch.float32)
    y_t = torch.tensor(train_df['click'].values, dtype=torch.float32)
    
    dqa_t = torch.tensor(train_df['dqa_present'].values, dtype=torch.float32) if use_dqa else None

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        
        # Forward pass
        predictions = model(pos_t, left_t, dqa_t)
        
        # Calculate loss and backpropagate
        loss = criterion(predictions, y_t)
        loss.backward()
        optimizer.step()
        
        print(f"Epoch {epoch+1}/{epochs} | Training Loss: {loss.item():.4f}")
    
    return model

def evaluate_pal(model, test_df, use_dqa=False):
    model.eval()
    with torch.no_grad():
        pos_t = torch.tensor(test_df['position'].values, dtype=torch.long)
        left_t = torch.tensor(test_df['is_left_column'].astype(int).values, dtype=torch.float32)
        y_true = test_df['click'].values
        
        dqa_t = torch.tensor(test_df['dqa_present'].values, dtype=torch.float32) if use_dqa else None
        
        probs = model(pos_t, left_t, dqa_t).numpy()
        
        auc = roc_auc_score(y_true, probs)
        loss = log_loss(y_true, probs)
        
        return auc, loss

In [ ]:
print("=========================================")
print("Training Standard PAL Baseline")
print("=========================================")
pal_standard = PALModel(max_position=train_imp['position'].max(), use_dqa_feature=False)
pal_standard = train_pal_model(pal_standard, train_imp, epochs=10, lr=0.01)

std_auc, std_loss = evaluate_pal(pal_standard, test_imp)
print(f"\nFinal Standard PAL Performance -> ROC-AUC: {std_auc:.4f} | LogLoss: {std_loss:.4f}\n")


print("=========================================")
print("Training DQA-Aware PAL Model")
print("=========================================")
pal_dqa = PALModel(max_position=dqa_train['position'].max(), use_dqa_feature=True)
pal_dqa = train_pal_model(pal_dqa, dqa_train, epochs=10, lr=0.01, use_dqa=True)

dqa_auc, dqa_loss = evaluate_pal(pal_dqa, dqa_test, use_dqa=True)
print(f"\nFinal DQA-Aware PAL Performance -> ROC-AUC: {dqa_auc:.4f} | LogLoss: {dqa_loss:.4f}")